# Training Arabic Sentiment Model

Arabic customer reviews, 3-class (Positive / Negative / Neutral).

Classical NLP: `preprocess_arabic` → word / char / FeatureUnion TF-IDF → NB / LR / Linear SVM with `class_weight='balanced'`.

Neutral is rare (~5%). Selection metric: **macro-F1 on validation**. Neutral is oversampled **after** the train/test split, on the fit/train fold only.

## 1. Imports

In [1]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

ROOT = Path("..") if (Path("..") / "data").exists() else Path(".")
sys.path.insert(0, str(ROOT.resolve()))

from src.Arabic_model import (
    ARABIC_FEATURE_NAMES,
    ARABIC_MODEL_NAMES,
    ArabicSentimentModel,
    build_arabic_baseline,
    build_arabic_pipeline,
    make_classifier,
    make_vectorizer,
    save_arabic_model,
)
from src.labels import ARABIC, NEGATIVE, NEUTRAL, POSITIVE, map_sentiment
from src.Preprocessing_pipeline import preprocess_arabic

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:.4f}".format)
LABELS = [NEGATIVE, NEUTRAL, POSITIVE]
print("مش kept:", "مش" in preprocess_arabic("الخدمة مش حلوة").split())
print("لا kept:", "لا" in preprocess_arabic("المنتج لا يستحق المال").split())

مش kept: True
لا kept: True


## 2. Load Data

In [2]:
DATA = ROOT / "data" / "raw" / "arabic" / "Final_Data.csv"
df = pd.read_csv(DATA)
print(DATA.resolve())
df.head()

/home/abdlrhman/Courses/IEEE SSCS/NLP/Week1/Project1/data/raw/arabic/Final_Data.csv


,review_description,rating,company
0,رائع,positive,talbat
1,برنامج رائع جدا يساعد على تلبيه الاحتياجات بشكل اسرع,positive,talbat
2,التطبيق لا يغتح دائما بيعطيني لا يوجد اتصال بالشبكة..مع انه النت عندي تمام شو الحل??,negative,talbat
3,لماذا لا يمكننا طلب من ماكدونالدز؟,negative,talbat
4,البرنامج بيظهر كل المطاعم و مغلقه مع انها بتكون فاتحه بقاله كده اكتر من شهر,negative,talbat


## 3. EDA

In [3]:
print("shape:", df.shape)
print("columns:", list(df.columns))
print(df["rating"].value_counts())
print(df.isnull().sum())

fig, ax = plt.subplots(figsize=(5, 3))
df["rating"].value_counts().reindex(["positive", "negative", "neutral"]).plot(
    kind="bar", ax=ax, color=["#2ca02c", "#d62728", "#7f7f7f"], rot=0
)
ax.set_title("Arabic class counts")
plt.tight_layout()
plt.close()

shape: (40046, 3)
columns: ['review_description', 'rating', 'company']
rating
positive    23921
negative    14200
neutral      1925
Name: count, dtype: int64
review_description    1
rating                0
company               0
dtype: int64


## 4. Cleaning

In [4]:
df = df.dropna(subset=["review_description"]).copy()
df["label"] = df["rating"].map(lambda x: map_sentiment(x, ARABIC))
conflict = df.groupby("review_description")["label"].nunique().loc[lambda s: s > 1].index
print("conflicting texts:", len(conflict))
df = df[~df["review_description"].isin(conflict)]
df = df.drop_duplicates(subset=["review_description"])
print(df["label"].value_counts())

conflicting texts: 95
label
Positive    23119
Negative    13900
Neutral      1889
Name: count, dtype: int64


## 5. Preprocessing

Same `preprocess_arabic` used at inference. Negation particles are kept.

In [5]:
df["clean_text"] = df["review_description"].map(preprocess_arabic)
df = df[df["clean_text"].str.len() > 0]
print("مش kept:", "مش" in preprocess_arabic("الخدمة مش حلوة").split())
df[["review_description", "clean_text", "label"]].head(3)

مش kept: True


,review_description,clean_text,label
0,رائع,رائع,Positive
1,برنامج رائع جدا يساعد على تلبيه الاحتياجات بشكل اسرع,برنامج رائع جدا يساعد علي تلبيه الاحتياجات بشكل اسرع,Positive
2,التطبيق لا يغتح دائما بيعطيني لا يوجد اتصال بالشبكة..مع انه النت عندي تمام شو الحل??,التطبيق لا يغتح دائما بيعطيني لا يوجد اتصال بالشبكة مع انه النت عندي تمام شو الحل,Negative


## 6. Train/Test Split

Oversample Neutral on the **training fold only**.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

def oversample_neutral(X, y, random_state=42):
    X = pd.Series(X).reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)
    target = int(y.value_counts().drop(NEUTRAL, errors="ignore").min())
    parts_x, parts_y = [], []
    for label in y.unique():
        Xi, yi = X[y == label], y[y == label]
        if label == NEUTRAL and len(Xi) < target:
            Xi, yi = resample(Xi, yi, replace=True, n_samples=target, random_state=random_state)
        parts_x.append(Xi)
        parts_y.append(yi)
    Xo = pd.concat(parts_x, ignore_index=True)
    yo = pd.concat(parts_y, ignore_index=True)
    perm = np.random.RandomState(random_state).permutation(len(Xo))
    return Xo.iloc[perm], yo.iloc[perm]

X_fit_os, y_fit_os = oversample_neutral(X_fit, y_fit)
print("train", len(X_train), "fit_os", len(X_fit_os), "val", len(X_val), "test", len(X_test))
print("val label counts\n", y_val.value_counts())

train 30603 fit_os 32118 val 6121 test 7651
val label counts
 label
Positive    3611
Negative    2210
Neutral      300
Name: count, dtype: int64


## 7. Feature Extraction

Word TF-IDF, word n-grams (1–2), char_wb 3–5, and a FeatureUnion of word+char. Word n-grams keep `لا يستحق`.

## 8. Baseline Model

In [7]:
def scores(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

baseline = build_arabic_baseline()
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)
print(classification_report(y_test, base_pred, labels=LABELS, digits=4, zero_division=0))
print(pd.Series(scores(y_test, base_pred), name="baseline"))

              precision    recall  f1-score   support

    Negative     0.8092    0.8001    0.8047      2762
     Neutral     0.1632    0.3324    0.2189       376
    Positive     0.8883    0.8176    0.8515      4513

    accuracy                         0.7875      7651
   macro avg     0.6202    0.6501    0.6250      7651
weighted avg     0.8241    0.7875    0.8035      7651

Accuracy    0.7875
Precision   0.6202
Recall      0.6501
F1          0.6250
Name: baseline, dtype: float64


## 9. Model Experiments

Grid fitted on oversampled `X_fit`. Scored on untouched `X_val`. Winner chosen by macro-F1.

In [8]:
rows = []
for feature_name in ARABIC_FEATURE_NAMES:
    vec = make_vectorizer(feature_name)
    Xtr = vec.fit_transform(X_fit_os)
    Xva = vec.transform(X_val)
    for model_name in ARABIC_MODEL_NAMES:
        clf = make_classifier(model_name)
        clf.fit(Xtr, y_fit_os)
        pred = clf.predict(Xva)
        row = {"Model": model_name, "Features": feature_name, **scores(y_val, pred)}
        rows.append(row)
        print(f"val {model_name:20} {feature_name:22} macro-F1={row['F1']:.4f}")

val_results = pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)
val_results

val Naive Bayes          Word TF-IDF            macro-F1=0.6264


val Logistic Regression  Word TF-IDF            macro-F1=0.6302


val Linear SVM           Word TF-IDF            macro-F1=0.6175


val Naive Bayes          Word n-grams           macro-F1=0.6218


val Logistic Regression  Word n-grams           macro-F1=0.6316


...

val Linear SVM           Character n-grams      macro-F1=0.6282


val Naive Bayes          Word + char n-grams    macro-F1=0.6311


val Logistic Regression  Word + char n-grams    macro-F1=0.6447


val Linear SVM           Word + char n-grams    macro-F1=0.6116


,Model,Features,Accuracy,Precision,Recall,F1
0,Logistic Regression,Word + char n-grams,0.8289,0.6421,0.6477,0.6447
1,Logistic Regression,Character n-grams,0.7935,0.6359,0.6815,0.6428
2,Logistic Regression,Word n-grams,0.8180,0.6313,0.6320,0.6316
3,Naive Bayes,Word + char n-grams,0.7914,0.6232,0.6572,0.6311
4,Logistic Regression,Word TF-IDF,0.7914,0.6263,0.6553,0.6302
5,Linear SVM,Character n-grams,0.8000,0.6229,0.6443,0.6282
6,Naive Bayes,Word TF-IDF,0.7858,0.6335,0.6534,0.6264
7,Naive Bayes,Character n-grams,0.7665,0.6237,0.6722,0.6227
8,Naive Bayes,Word n-grams,0.8280,0.6385,0.6123,0.6218
9,Linear SVM,Word TF-IDF,0.7989,0.6169,0.6242,0.6175


## 10. Evaluation

In [9]:
best_row = val_results.iloc[0]
best_model_name, best_feature_name = best_row["Model"], best_row["Features"]
print("validation winner:", best_model_name, "+", best_feature_name)

X_train_os, y_train_os = oversample_neutral(X_train, y_train)
best = build_arabic_pipeline(best_feature_name, best_model_name)
best.fit(X_train_os, y_train_os)
y_pred = best.predict(X_test)
print("\nHeld-out TEST")
print(classification_report(y_test, y_pred, labels=LABELS, digits=4, zero_division=0))
print(pd.Series(scores(y_test, y_pred), name="test"))

cm = confusion_matrix(y_test, y_pred, labels=LABELS)
print("Confusion matrix labels", LABELS)
print(cm)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(cmap="Blues", ax=ax, colorbar=False)
ax.set_title("Arabic — test confusion matrix")
plt.tight_layout()
plt.close()

validation winner: Logistic Regression + Word + char n-grams



Held-out TEST
              precision    recall  f1-score   support

    Negative     0.7983    0.8385    0.8179      2762
     Neutral     0.1681    0.2048    0.1847       376
    Positive     0.8917    0.8480    0.8693      4513

    accuracy                         0.8130      7651
   macro avg     0.6194    0.6304    0.6240      7651
weighted avg     0.8224    0.8130    0.8171      7651

Accuracy    0.8130
Precision   0.6194
Recall      0.6304
F1          0.6240
Name: test, dtype: float64
Confusion matrix labels ['Negative', 'Neutral', 'Positive']
[[2316  122  324]
 [ 158   77  141]
 [ 427  259 3827]]


Held-out test confusion matrix (`Negative`, `Neutral`, `Positive`):

|  | Pred Negative | Pred Neutral | Pred Positive |
| :--- | ---: | ---: | ---: |
| **True Negative** | 2316 | 122 | 324 |
| **True Neutral** | 158 | 77 | 141 |
| **True Positive** | 427 | 259 | 3827 |

Accuracy **0.8130**. Macro-F1 **0.6240**. Neutral F1 **0.1847** — Neutral is rare and noisy; accuracy overstates quality.

## Error analysis

In [10]:
tmp = ArabicSentimentModel(best)
for text in (
    "الفيلم مش وحش",
    "مش حلو",
    "المنتج لا يستحق المال",
    "الخدمة ممتازة والتجربة كانت رائعة",
    "الفيلم سيء جدا",
):
    print(f"{tmp.predict(text):10}  {text}")

Negative    الفيلم مش وحش
Neutral     مش حلو
Negative    المنتج لا يستحق المال
Positive    الخدمة ممتازة والتجربة كانت رائعة
Negative    الفيلم سيء جدا


## 11. Best Model

In [11]:
print(best)
print(best_row.to_dict())

Pipeline(steps=[('vec',
                 FeatureUnion(transformer_list=[('word',
                                                 TfidfVectorizer(max_features=20000,
                                                                 ngram_range=(1,
                                                                              2))),
                                                ('char',
                                                 TfidfVectorizer(analyzer='char_wb',
                                                                 max_features=20000,
                                                                 ngram_range=(3,
                                                                              5)))])),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=2000))])
{'Model': 'Logistic Regression', 'Features': 'Word + char n-grams', 'Accuracy': 0.8289495180526057, 'Precision': 0.6421120617220838, 'Recall': 0.6476778582397036, 'F1': 0.

## 12. Save Model

In [12]:
path = save_arabic_model(best, ROOT / "models" / "Arabic_model_weights.pkl")
print("saved", path.resolve())

saved /home/abdlrhman/Courses/IEEE SSCS/NLP/Week1/Project1/models/Arabic_model_weights.pkl


## 13. Example Predictions

In [13]:
model = ArabicSentimentModel.load(path)
for text in (
    "الفيلم رائع جدا",
    "الخدمة ممتازة",
    "أنا سعيد جدا بالمنتج",
    "أنا أحب هذا المنتج",
    "الفيلم سيء جدا",
    "الخدمة كانت سيئة",
    "الخدمة سيئة للغاية",
    "المنتج لا يستحق المال",
):
    print(f"{model.predict(text):10}  {text}")

Positive    الفيلم رائع جدا
Positive    الخدمة ممتازة
Positive    أنا سعيد جدا بالمنتج
Positive    أنا أحب هذا المنتج
Negative    الفيلم سيء جدا
Negative    الخدمة كانت سيئة
Negative    الخدمة سيئة للغاية
Negative    المنتج لا يستحق المال
